<a href="https://colab.research.google.com/github/NgocAnhNguyen-0601/Housing-Price-Prediction-/blob/main/Housing_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Housing Price Prediction

# Dataset: California Housing Price

Process: EDA -> Preprocessing -> Baseline model -> Model Selection -> Tuning -> Final Evaluation -> Inference

# 1. Set Up:

In [1]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

In [8]:
# Configuration:
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f"{x:.3f}")
sns.set_theme(style='darkgrid')

plt.rcParams.update({
    "axes.titlesize":10,
    "axes.labelsize":9,
    "xtick.labelsize":8,
    "ytick.labelsize":8
})

RANDOM_STATE=42
CSV_PATH = "/content/Housing-Price-Prediction-/housing.csv"   # update paht for a different dataset.
TARGET_COL = "median_house_value" # Target Column Value

# 2. Load Data

In [7]:
!git clone https://github.com/NgocAnhNguyen-0601/Housing-Price-Prediction-.git

Cloning into 'Housing-Price-Prediction-'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 9 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 402.24 KiB | 1022.00 KiB/s, done.


In [9]:
df = pd.read_csv(CSV_PATH)

In [10]:
print("Data frame Shape:", df.shape)

Data frame Shape: (20640, 10)


In [11]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.230,37.880,41.000,880.000,129.000,322.000,126.000,8.325,452600.000,NEAR BAY
1,-122.220,37.860,21.000,7099.000,1106.000,2401.000,1138.000,8.301,358500.000,NEAR BAY
2,-122.240,37.850,52.000,1467.000,190.000,496.000,177.000,7.257,352100.000,NEAR BAY
3,-122.250,37.850,52.000,1274.000,235.000,558.000,219.000,5.643,341300.000,NEAR BAY
4,-122.250,37.850,52.000,1627.000,280.000,565.000,259.000,3.846,342200.000,NEAR BAY


3. Exploratory Data Analysis (EDA)

In [12]:
df.columns

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value', 'ocean_proximity'],
      dtype='object')

# Column details:
1. longtitude: A measure of how dar west a house is, a higher value is farther west
2. latitude: A measure of how far north a house is, a higher value is farther north
3. Housing Median Age: Median age of a house within a bloack, a lower number is a newer building.
4. Total Room: Total number of rooms within a block.
5. Total Bedrooms: Total number of bedrooms within a block.
6. Population: Total number of people residing within a block.
7. Households: Total number of households, a group of peopole residing within a home unit, for a block.
8. Median income: Median income for households within a block of houses (measured in tens of thousands of US dollars)
9. Median House Value: Median House Value for housegolds within a block (measured in UD Dollar)
10. Ocean Proximity: Location of the hosuen within ocean/sea

In [13]:
# Basic dataset overview:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [15]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Target Column:", TARGET_COL)
print("Numerical Columns:", num_cols)
print("Categorical Columns:", cat_cols)

Target Column: median_house_value
Numerical Columns: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']
Categorical Columns: ['ocean_proximity']
